# Keysight Customer Journey Analysis Demo

This notebook loads synthetic Keysight-style lead data, builds a Markov-chain-based view of customer journeys, and prepares a transitions dataset for Amazon QuickSight Sankey visualizations.


In [ ]:
import pandas as pd
import ast

combined_path = 'keysight_synthetic_combined_25000.csv'
df = pd.read_csv(combined_path)
df.head()


In [ ]:
from collections import Counter

transitions = Counter()
for seq_str in df['Touchpoint_Sequence'].dropna():
    try:
        seq = ast.literal_eval(seq_str)
    except Exception:
        continue
    if not isinstance(seq, list):
        continue
    for i in range(len(seq) - 1):
        pair = (seq[i], seq[i+1])
        transitions[pair] += 1

transitions_df = pd.DataFrame([
    {'from_step': k[0], 'to_step': k[1], 'count': v}
    for k, v in transitions.items()
])
transitions_df.head()


In [ ]:
row_sums = transitions_df.groupby('from_step')['count'].transform('sum')
transitions_df['probability'] = transitions_df['count'] / row_sums
transitions_df.sort_values(['from_step','probability'], ascending=[True, False]).head(20)


In [ ]:
transitions_output = 'keysight_transitions_for_quicksight.csv'
transitions_df.to_csv(transitions_output, index=False)
transitions_output
